# DeepSpeech2 ASR — demo

Inference of the final model (DeepSpeech2, 14.45M parameters, trained on LibriSpeech 960h)
with a 3-gram LM (beam search + KenLM), and WER/CER computation.

**Run on GPU:** Runtime → Change runtime type → T4 GPU. Then Runtime → Run all.

The notebook:
1. clones the repository and installs the requirements (Python 3.12 environment, see below);
2. downloads the checkpoint and the LM;
3. downloads an example dataset (a zip archive on Google Drive), runs `inference.py` on it and `calc_metrics.py`.

To check the model on **your own data**, put a Google Drive link to your archive into `DATASET_URL` (step 4) and run steps 4–5 again.

## 1. Clone the repository

In [ ]:
!git clone https://github.com/zuvirre/dla-hw1-asr.git
%cd /content/dla-hw1-asr

## 2. Install the requirements

The project uses `torch==2.2.0` and `numpy==1.26.4`, which support Python up to 3.12, while Colab runs Python 3.13.
So we create a separate environment with Python 3.12 using [uv](https://docs.astral.sh/uv/) (it downloads Python 3.12 itself)
and install `requirements.txt` into it. All scripts below are run with this environment's interpreter `/content/venv/bin/python`.

KenLM is built from source (`requirements.txt`), it takes about a minute.

In [ ]:
!pip install -q uv
!uv venv --python 3.12 /content/venv
!uv pip install --python /content/venv/bin/python -r requirements.txt

In [ ]:
# Colab sets MPLBACKEND to its inline backend, which is not installed in our
# environment -> use the non-interactive Agg backend for the scripts
%env MPLBACKEND=Agg

# check the environment: versions and GPU availability
!/content/venv/bin/python -c "import torch, numpy, kenlm, pyctcdecode; print('torch', torch.__version__, '| numpy', numpy.__version__, '| CUDA', torch.cuda.is_available())"

## 3. Download the checkpoint and the LM

`download_weights.py` downloads the final checkpoint to `saved/ds2_final2_960h/model_best.pth`
and the lowercase LibriSpeech 3-gram LM to `data/lm/3-gram.pruned.1e-7.lower.arpa` — the paths used in `src/configs/inference.yaml`.

In [ ]:
!/content/venv/bin/python download_weights.py

## 4. Download a dataset

`DATASET_URL` is a Google Drive link (access: "Anyone with the link") to a **folder** or to a **zip archive** of a folder with the structure

```
NameOfTheDirectoryWithUtterances
├── audio
│   ├── UtteranceID1.wav  # .wav, .flac or .mp3
│   └── ...
└── transcriptions        # optional, needed for metrics
    ├── UtteranceID1.txt
    └── ...
```

The default link is a small example: 5 utterances from LibriSpeech test-clean (CC BY 4.0).
Replace it with a link to your own folder or archive to check the model on your data.
(A zip archive is more reliable for large datasets: Google Drive may limit downloading folders with many files.)

In [ ]:
DATASET_URL = "https://drive.google.com/file/d/1hq89ew9UDJoJd014mFqHLUPkZ92mYdcO/view?usp=sharing"

from pathlib import Path

!rm -rf /content/custom_data && mkdir -p /content/custom_data
if "/folders/" in DATASET_URL:  # link to a Google Drive folder
    !/content/venv/bin/gdown --folder "{DATASET_URL}" -O /content/custom_data
else:  # link to a zip archive
    !/content/venv/bin/gdown "{DATASET_URL}" -O /content/custom_data/data.zip
    !unzip -q /content/custom_data/data.zip -d /content/custom_data

# folder that contains audio/ (the archive may have a top-level folder or not)
DATA_DIR = next(
    p.parent for p in Path("/content/custom_data").rglob("audio")
    if p.is_dir() and "__MACOSX" not in p.parts
)
print(DATA_DIR, *sorted(p.name for p in DATA_DIR.iterdir()))

## 5. Inference and metrics

`inference.py` writes a prediction `<UtteranceID>.txt` for every audio file to `/content/predictions`
(decoder: beam search + LM, beam size 10). If the dataset has transcriptions, it also prints WER/CER
for three decoders: argmax, beam search, beam search + LM.

`calc_metrics.py` computes WER/CER from the saved predictions and the ground-truth transcriptions.

In [ ]:
!/content/venv/bin/python inference.py datasets=custom_dir datasets.custom.data_dir={DATA_DIR} inferencer.save_path=/content/predictions

In [ ]:
if (DATA_DIR / "transcriptions").exists():
    !/content/venv/bin/python calc_metrics.py --gt_dir {DATA_DIR}/transcriptions --pred_dir /content/predictions
else:
    print("No transcriptions in the dataset - metrics are not computed")

In [ ]:
# first N_EXAMPLES predictions next to the ground truth (if available)
N_EXAMPLES = 10
preds = sorted(Path("/content/predictions").glob("*.txt"))
print(f"Showing {min(N_EXAMPLES, len(preds))} of {len(preds)} predictions")
for pred in preds[:N_EXAMPLES]:
    print(pred.stem)
    gt = DATA_DIR / "transcriptions" / pred.name
    if gt.exists():
        print("  GT:  ", gt.read_text().strip())
    print("  PRED:", pred.read_text().strip())